# 05 · Fit the fuser, calibrate, run the acceptance gate

This notebook produces the two things the deployed app actually reads —
`fusion_a.json` and `fusion_b.json` — and the honest evaluation report.

**Calibration is not optional (A6).** A reported "82% AI" must correspond to
an observed 82% positive rate within 5 points. Uncalibrated confidence is the
single most common failure of tools in this category, and it is what makes a
score usable as evidence rather than as a vibe.

**A4 is a release blocker.** Detectors are known to over-flag non-native
English writers. If the ESL false-positive rate exceeds 5%, this build does
not ship, regardless of how good every other number looks.


In [ ]:
# Kaggle setup. Run once per session.
!pip install -q "transformers>=4.44" "datasets>=2.20" sentencepiece onnx onnxruntime \
    "optimum[onnxruntime]" pyarrow

import sys, os
from pathlib import Path

# The repo is added as a Kaggle dataset, or cloned. Point REPO at it.
REPO = Path("/kaggle/input/ai-detector-repo") if Path("/kaggle/input/ai-detector-repo").exists() \
       else Path("/kaggle/working/ai-detector")
if not REPO.exists():
    !git clone --depth 1 $GIT_URL /kaggle/working/ai-detector

sys.path.insert(0, str(REPO / "training"))
sys.path.insert(0, str(REPO / "api"))

WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
DATA = WORK / "data"; DATA.mkdir(exist_ok=True)
MODELS = WORK / "models"; MODELS.mkdir(exist_ok=True)
print("repo:", REPO)


In [ ]:
import numpy as np, pandas as pd, json
from pathlib import Path
from lib.calibrate import (calibration_error, fit_fusion, fit_isotonic,
                           run_acceptance_gate, write_fusion_config)

# Each split must be genuinely held out (PRD 12.4). Replace these loaders with
# your own paths — RAID-test, M4, an essay set, and an ESL/ELL corpus.
def load_split(name):
    """Return (signals[N,4], labels[N]). Signals in the order:
    classifier, features, binoculars_ratio, burstiness."""
    path = DATA / f"eval_{name}.parquet"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} is missing. Score each held-out split with the exported "
            "ONNX models plus api/_lib/stats.py and save the four signals."
        )
    frame = pd.read_parquet(path)
    signals = frame[["classifier", "features", "binoculars_ratio", "burstiness"]].to_numpy(np.float64)
    return signals, frame["label"].to_numpy(np.float64)


In [ ]:
# Fit the fuser on a calibration split, never on anything used for training
# or for the final report.
for tag in ("a", "b"):
    signals, labels = load_split(f"calibration_{tag}")
    weights, bias = fit_fusion(signals, labels)
    print(f"Model {tag.upper()} weights:",
          dict(zip(["classifier","features","binoculars","burstiness"], weights.round(3))),
          f"bias {bias:.3f}")

    from lib.calibrate import _sigmoid, _logit
    raw = _sigmoid(_logit(signals) @ weights + bias)

    before = calibration_error(raw, labels)
    x, y = fit_isotonic(raw, labels)
    after = calibration_error(np.interp(raw, x, y), labels)
    print(f"  worst decile gap: {before['max_gap_points']:.1f} -> {after['max_gap_points']:.1f} points")

    write_fusion_config(MODELS / f"fusion_{tag}.json", weights, bias, x, y, version=f"{tag}-1.0.0")


In [ ]:
# The acceptance gate (PRD 14). Every number here goes in the published report.
def fused(tag, split):
    config = json.loads((MODELS / f"fusion_{tag}.json").read_text())
    signals, labels = load_split(split)
    from lib.calibrate import _sigmoid, _logit
    raw = _sigmoid(_logit(signals) @ np.asarray(config["weights"]) + config["bias"])
    return np.interp(raw, config["calibration_x"], config["calibration_y"]), labels

manifest = json.loads((MODELS / "manifest.json").read_text())

report = run_acceptance_gate(
    in_domain=fused("a", "in_domain"),
    cross_domain=fused("a", "m4"),
    essays=fused("a", "essays"),
    esl=fused("a", "esl"),
    humanized=fused("a", "humanized"),
    calibrated=fused("a", "in_domain"),
    onnx_max_logit_delta=manifest["parity"]["a"]["max_logit_delta"],
    bundle_sizes_mb=manifest["bundle_sizes_mb"],
)

print(report.summary())
report.write(MODELS / "eval_report.json")


In [ ]:
# A4 is a release blocker. This cell is meant to stop the pipeline.
blockers = report.blockers_failed
if blockers:
    raise SystemExit(
        "DO NOT SHIP. Blocking criteria failed: " + ", ".join(c.id for c in blockers) +
        "\n\nA4 exists because detectors are known to over-flag non-native English "
        "writers. Shipping a tool that penalises ESL authors is a real harm, not a "
        "metric regression. Retrain with more ESL data in the negative class and a "
        "lower alpha in the partial-AUROC loss before trying again."
    )
print("No blocking criteria failed.")


In [ ]:
# A7 and A8 come from the humanizer, not the detector. Run them against the
# deployed app once the models are live:
#
#   A7  surrogate score > 0.9 falls below 0.3 within 3 passes, >= 90% of docs
#   A8  meaning similarity retained >= 0.85
#
# scripts/eval_humanizer.py drives the deployed endpoint over a sample and
# writes those two numbers into the same report.
print(open(MODELS / "eval_report.json").read())
